# maxpool-reduce — worked example 3: 1-D max pooling via einops reduce

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `maxpool-reduce`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The reduce-as-pool idea is not 2-D specific. For a `(B, C, L)` signal, `reduce(x, 'b c (l p) -> b c l', 'max', p=p)` factors the single length axis and maxes over each window — a 1-D max pool equivalent to `F.max_pool1d`.

## Worked solution

We apply the same pattern to a 1-D length axis.

1. **Factor length.** `(l p)` splits `L` into `l * p`; with `p` the window size each group of `p` consecutive samples is one window.
2. **Max reduce.** Output `b c l` drops `p`, taking the per-window maximum.
3. **Equivalence.** The result matches `F.max_pool1d(x, kernel_size=p)`.

The demo pools a `(1, 2, 8)` signal with `p=4` into `(1, 2, 2)` and confirms it equals the torch reference.

In [ ]:
import torch as t
import torch.nn.functional as F
import einops

t.manual_seed(2)

def maxpool1d_via_reduce(x, p):
    return einops.reduce(x, 'b c (l p) -> b c l', 'max', p=p)

x = t.randn(1, 2, 8)
out = maxpool1d_via_reduce(x, 4)
ref = F.max_pool1d(x, kernel_size=4)
print('out shape:', tuple(out.shape))
print('matches F.max_pool1d:', t.allclose(out, ref))